# Colab Drive
Hệ thống tự động hóa tải và đồng bộ tệp tin đa nguồn vào Google Drive trên nền tảng Google Colab.

Tích hợp đa công cụ tải xuống chuyên dụng:
- yt-dlp: Tải video từ hơn 1.800 nền tảng trực tuyến (YouTube, Facebook, TikTok, Bilibili, VK, OK.ru...) và luồng stream HLS/DASH kèm tiến trình thời gian thực (ETA).
- Deno JavaScript Runtime: Tự động giải mã chữ ký YouTube (n-sig/cipher) chống lỗi 403 Forbidden.
- aria2c: Tăng tốc tải đa luồng cho các tệp trực tiếp (Direct Link) và giao thức chia sẻ P2P BitTorrent/Magnet link.
- gdown: Chuyên dụng cho Google Drive (sao chép tệp đơn lẻ vượt cảnh báo virus hoặc toàn bộ thư mục Drive sang Drive cá nhân).
- megatools: Tải tệp tin và gói lưu trữ từ nền tảng MEGA (mega.nz).
- p7zip / unrar: Tự động giải nén tệp lưu trữ đa định dạng ngay trên đám mây.

Quy trình vận hành 4 bước chuẩn hóa:
- Bước 1: Thiết lập môi trường, cài đặt các công cụ tải, Deno JS runtime và kích hoạt chống ngắt kết nối (Anti-Disconnect).
- Bước 2: Xác thực liên kết phân vùng Google Drive và kiểm tra dung lượng bộ đệm máy ảo.
- Bước 3: Nhập liên kết và kiểm tra trước thông số nguồn (Hiển thị rõ 1080p, 2K, 4K, dung lượng, thời lượng).
- Bước 4: Cấu hình lưu trữ và tiến hành tải xuống đồng bộ vào Google Drive.


In [ ]:
#@title Bước 1: Thiết lập môi trường và cài đặt các công cụ tải chuyên dụng
#@markdown Cài đặt hệ sinh thái công cụ: yt-dlp, Deno JS runtime, gdown, megatools, aria2c, ffmpeg, p7zip và kích hoạt chống ngắt kết nối.

import os
import sys
from datetime import datetime, timezone, timedelta
from IPython.display import display, Javascript

VN_TZ = timezone(timedelta(hours=7))

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "PROBE": "\033[95m",
        "PROGRESS": "\033[96m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

log("SYSTEM", "Bắt đầu khởi tạo môi trường hệ thống...", "SYSTEM")

display(Javascript('''
function KeepClicking(){
   console.log("[Colab Drive] Anti-disconnect heartbeat pulse sent");
   document.querySelector("colab-connect-button")?.shadowRoot?.querySelector("#connect")?.click();
}
setInterval(KeepClicking, 60000);
'''))
log("SYSTEM", "Đã kích hoạt cơ chế tự động duy trì kết nối phiên làm việc (Anti-Disconnect).", "SYSTEM")

log("SYSTEM", "Cài đặt và cập nhật các gói Python: yt-dlp, gdown, streamlink...", "INFO")
os.system("pip install --upgrade --no-cache-dir -q yt-dlp gdown streamlink")

log("SYSTEM", "Cài đặt Deno JavaScript Runtime (giải mã chữ ký YouTube n-sig chống 403 Forbidden)...", "INFO")
os.system("curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1")
deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.exists(deno_bin) and deno_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{deno_bin}:{os.environ['PATH']}"

log("SYSTEM", "Cài đặt các công cụ hệ thống: aria2c, ffmpeg, megatools, p7zip-full, unrar, curl, wget, nodejs...", "INFO")
os.system("apt-get update -y -q && apt-get install -y -q ffmpeg aria2 megatools p7zip-full unrar curl wget nodejs")

log("SUCCESS", "Hệ thống các công cụ tải chuyên dụng và Deno JS runtime đã sẵn sàng.", "SUCCESS")



In [ ]:
#@title Bước 2: Xác thực và kết nối Google Drive
#@markdown Liên kết phân vùng lưu trữ Google Drive và kiểm tra bộ đệm máy ảo xử lý tệp.

import os
import shutil
from datetime import datetime, timezone, timedelta

VN_TZ = timezone(timedelta(hours=7))

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "PROBE": "\033[95m",
        "PROGRESS": "\033[96m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

log("DRIVE", "Yêu cầu quyền truy cập phân vùng Google Drive...", "DRIVE")
try:
    from google.colab import drive
    drive.mount('/content/drive')
    if os.path.exists('/content/drive/MyDrive'):
        log("SUCCESS", "Kết nối Google Drive thành công.", "SUCCESS")
        log("DRIVE", "Điểm gắn lưu trữ: /content/drive/MyDrive/", "INFO")
        
        total, used, free = shutil.disk_usage('/content')
        total_gb = total / (1024**3)
        used_gb = used / (1024**3)
        free_gb = free / (1024**3)
        log("SYSTEM", f"Bộ đệm máy ảo Colab: Đã dùng {used_gb:.2f} GB / Khả dụng {free_gb:.2f} GB (Tổng: {total_gb:.2f} GB).", "INFO")
        if free_gb < 2.0:
            log("WARN", f"Cảnh báo: Bộ đệm máy ảo Colab còn dưới {free_gb:.2f} GB khả dụng.", "WARN")
    else:
        log("WARN", "Đã gắn kết phân vùng nhưng chưa nhận diện được thư mục MyDrive.", "WARN")
except Exception as e:
    log("ERROR", f"Kết nối Google Drive thất bại: {e}", "ERROR")



In [ ]:
#@title Bước 3: Nhập liên kết và kiểm tra thông tin nguồn
#@markdown Hỗ trợ mọi nguồn: Google Drive (tệp/thư mục), MEGA, BitTorrent/Magnet, Video trực tuyến (YouTube, TikTok...) và Direct Link.

LINK_CAN_TAI = "" #@param {type:"string"}
DANH_SACH_PROXY = "" #@param {type:"string"}
REFERER_VUOT_CHAN = "" #@param {type:"string"}
NOI_DUNG_COOKIE = "" #@param {type:"string"}

import os
import re
import json
import urllib.request
import urllib.parse
from datetime import datetime, timezone, timedelta
import yt_dlp

VN_TZ = timezone(timedelta(hours=7))

deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.exists(deno_bin) and deno_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{deno_bin}:{os.environ['PATH']}"

def log(tag, msg, level="INFO"):
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "PROBE": "\033[95m",
        "PROGRESS": "\033[96m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

def parse_proxies(proxy_raw):
    proxies = []
    if not proxy_raw or not proxy_raw.strip():
        return proxies
    raw_clean = proxy_raw.replace(',', '\n').replace(';', '\n').replace('|', '\n').replace(' ', '\n')
    for line in raw_clean.split('\n'):
        line = line.strip()
        if not line:
            continue
        parts = line.split(':')
        if len(parts) == 4:
            ip, port, user, pwd = parts
            proxies.append(f"http://{user}:{pwd}@{ip}:{port}")
        elif len(parts) == 2:
            ip, port = parts
            proxies.append(f"http://{ip}:{port}")
        elif line.startswith(('http://', 'https://', 'socks5://', 'socks4://')):
            proxies.append(line)
    return proxies

def format_size(bytes_val):
    if not bytes_val or bytes_val <= 0:
        return "Không xác định"
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes_val < 1024.0:
            return f"{bytes_val:.2f} {unit}"
        bytes_val /= 1024.0
    return f"{bytes_val:.2f} PB"

def format_resolution(width, height, fps=None):
    if not height:
        return "Tự động (Chất lượng cao nhất)"
    try:
        h = int(height)
        w = int(width) if width else 0
        if h >= 4320 or w >= 7680:
            label = "8K Ultra HD (4320p)"
        elif h >= 2160 or w >= 3840:
            label = "4K Ultra HD (2160p)"
        elif h >= 1440 or w >= 2560:
            label = "2K QHD (1440p)"
        elif h >= 1080 or w >= 1920:
            label = "Full HD (1080p)"
        elif h >= 720 or w >= 1280:
            label = "HD (720p)"
        elif h >= 480:
            label = "SD (480p)"
        elif h >= 360:
            label = "360p"
        elif h >= 240:
            label = "240p"
        elif h >= 144:
            label = "144p"
        else:
            label = f"{h}p"
            
        dim_str = f"{w}x{h}" if w > 0 else f"{h}p"
        fps_str = f" @ {fps}fps" if fps and int(fps) > 30 else ""
        return f"{label} ({dim_str}{fps_str})"
    except:
        return f"{height}p"

if not LINK_CAN_TAI.strip():
    log("ERROR", "Chưa có thông tin liên kết đầu vào.", "ERROR")
    raise ValueError("Vui lòng nhập liên kết hợp lệ tại ô LINK_CAN_TAI.")

raw_urls = LINK_CAN_TAI.replace(',', '\n').replace(';', '\n')
urls = [u.strip() for u in raw_urls.split('\n') if u.strip()]

proxies = parse_proxies(DANH_SACH_PROXY)
temp_workspace = "/content/temp_download"
os.makedirs(temp_workspace, exist_ok=True)

cookie_file_path = None
if NOI_DUNG_COOKIE.strip():
    cookie_input = NOI_DUNG_COOKIE.strip()
    if os.path.isfile(cookie_input):
        cookie_file_path = cookie_input
        log("SYSTEM", f"Sử dụng tệp Cookie xác thực: {cookie_file_path}", "INFO")
    else:
        cookie_file_path = os.path.join(temp_workspace, "auth_cookies.txt")
        with open(cookie_file_path, "w", encoding="utf-8") as cf:
            cf.write(cookie_input)
        log("SYSTEM", "Đã khởi tạo tệp Cookie xác thực từ dữ liệu đầu vào.", "INFO")

log("SYSTEM", f"Bắt đầu phân tích thông số kỹ thuật cho {len(urls)} liên kết...", "SYSTEM")

session_tasks = []

for idx, url in enumerate(urls, start=1):
    log("PROBE", f"[{idx}/{len(urls)}] Phân tích liên kết: {url}", "PROBE")
    current_proxy = proxies[(idx - 1) % len(proxies)] if proxies else None
    
    is_torrent = url.startswith('magnet:?') or url.lower().endswith('.torrent') or '.torrent?' in url.lower()
    is_gdrive = 'drive.google.com' in url.lower() or 'docs.google.com' in url.lower()
    is_mega = 'mega.nz' in url.lower() or 'mega.co.nz' in url.lower()
    
    task_info = {
        "url": url,
        "type": "standard",
        "proxy": current_proxy,
        "title": "Tệp tin tải xuống",
        "resolution": "Không xác định",
        "filesize_str": "Không xác định",
        "duration_str": "Không xác định",
        "ext": "mp4"
    }
    
    if is_gdrive:
        task_info["type"] = "gdrive"
        is_folder = '/folders/' in url.lower()
        task_info["is_folder"] = is_folder
        task_info["resolution"] = "Thư mục Google Drive" if is_folder else "Tệp Google Drive"
        log("INFO", f"Nguồn: Google Drive ({task_info['resolution']})", "INFO")
        log("INFO", f"Công cụ tải tối ưu: gdown (Hỗ trợ vượt xác thực virus & tải trọn thư mục)", "INFO")
        session_tasks.append(task_info)
        continue

    elif is_mega:
        task_info["type"] = "mega"
        task_info["resolution"] = "Dữ liệu đám mây MEGA"
        log("INFO", f"Nguồn: Nền tảng lưu trữ MEGA (mega.nz)", "INFO")
        log("INFO", f"Công cụ tải tối ưu: megatools", "INFO")
        session_tasks.append(task_info)
        continue

    elif is_torrent:
        task_info["type"] = "torrent"
        task_info["title"] = "Gói tệp BitTorrent / Magnet"
        task_info["resolution"] = "Đa luồng P2P"
        log("INFO", f"Nguồn: Giao thức chia sẻ tệp BitTorrent / Magnet Link", "INFO")
        log("INFO", f"Công cụ tải tối ưu: aria2c (Kích hoạt DHT và OpenTrackers)", "INFO")
        session_tasks.append(task_info)
        continue

    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    }
    if REFERER_VUOT_CHAN.strip():
        headers['Referer'] = REFERER_VUOT_CHAN.strip()
        headers['Origin'] = urllib.parse.urlsplit(REFERER_VUOT_CHAN.strip()).scheme + "://" + urllib.parse.urlsplit(REFERER_VUOT_CHAN.strip()).netloc
    elif 'seenow' in url.lower() or 'vtv' in url.lower():
        headers['Referer'] = 'https://vtvprime.vn/'
        headers['Origin'] = 'https://vtvprime.vn'

    ydl_inspect_opts = {
        'quiet': True,
        'no_warnings': True,
        'http_headers': headers,
        'remote_components': ['ejs:github'],
        'extractor_args': {
            'youtube': {
                'player_client': ['web', 'mweb', 'android']
            }
        }
    }
    if cookie_file_path:
        ydl_inspect_opts['cookiefile'] = cookie_file_path
    if current_proxy:
        ydl_inspect_opts['proxy'] = current_proxy

    try:
        with yt_dlp.YoutubeDL(ydl_inspect_opts) as ydl:
            meta = ydl.extract_info(url, download=False)
            
            title = meta.get('title', 'Không xác định')
            ext = meta.get('ext', 'mp4')
            duration = meta.get('duration_string') or (f"{meta.get('duration')}s" if meta.get('duration') else "Không xác định")
            
            width = meta.get('width')
            height = meta.get('height')
            fps = meta.get('fps')
            resolution_label = format_resolution(width, height, fps)
                
            bytes_sz = meta.get('filesize') or meta.get('filesize_approx')
            size_str = format_size(bytes_sz)
            
            task_info.update({
                "type": "video_stream",
                "title": title,
                "resolution": resolution_label,
                "filesize_str": size_str,
                "duration_str": duration,
                "ext": ext
            })
            
            log("INFO", f"Tiêu đề: {title}", "INFO")
            log("INFO", f"Độ phân giải: {resolution_label} | Thời lượng: {duration}", "INFO")
            log("INFO", f"Dung lượng ước tính: {size_str} | Định dạng: {ext}", "INFO")
            
    except Exception as e:
        task_info["type"] = "direct_file"
        try:
            req = urllib.request.Request(url, headers=headers, method='HEAD')
            with urllib.request.urlopen(req, timeout=5) as resp:
                sz = int(resp.headers.get('Content-Length', 0))
                ctype = resp.headers.get('Content-Type', 'Direct link')
                task_info["title"] = os.path.basename(urllib.parse.urlsplit(url).path) or "direct_file"
                task_info["filesize_str"] = format_size(sz)
                task_info["resolution"] = f"Tệp tải trực tiếp ({ctype})"
                log("INFO", f"Tiêu đề: {task_info['title']}", "INFO")
                log("INFO", f"Dung lượng tệp: {task_info['filesize_str']} | Định dạng: {ctype}", "INFO")
        except:
            log("WARN", f"Không thể lấy trước siêu dữ liệu: {e}. Hệ thống sẽ xử lý bằng bộ tải đa luồng ở Bước 4.", "WARN")

    session_tasks.append(task_info)

cache_path = os.path.join(temp_workspace, "session_tasks.json")
with open(cache_path, "w", encoding="utf-8") as f:
    json.dump({
        "urls": urls,
        "proxies": proxies,
        "referer": REFERER_VUOT_CHAN.strip(),
        "cookie_file": cookie_file_path,
        "tasks": session_tasks
    }, f, ensure_ascii=False, indent=2)

print("\n" + "="*60)
log("SUCCESS", f"Hoàn tất kiểm tra {len(session_tasks)} liên kết. Sẵn sàng cấu hình và tải ở Bước 4.", "SUCCESS")
print("="*60)



In [ ]:
#@title Bước 4: Cấu hình lưu trữ và tiến hành tải xuống
#@markdown Thiết lập thư mục lưu trữ trên Google Drive, đặt tên tệp tùy biến và bắt đầu tiến trình tải đa nguồn kèm theo dõi thời gian thực.

THU_MUC_DRIVE = "/content/drive/MyDrive/colab-drive" #@param {type:"string"}
TEN_FILE_TU_DAT = "" #@param {type:"string"}
TU_DONG_GIAI_NEN = False #@param {type:"boolean"}
TELEGRAM_BOT_TOKEN = "" #@param {type:"string"}
TELEGRAM_CHAT_ID = "" #@param {type:"string"}

import os
import re
import sys
import time
import glob
import json
import shutil
import urllib.request
import urllib.parse
from datetime import datetime, timezone, timedelta
import yt_dlp

VN_TZ = timezone(timedelta(hours=7))

deno_bin = os.path.expanduser("~/.deno/bin")
if os.path.exists(deno_bin) and deno_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{deno_bin}:{os.environ['PATH']}"

is_progress_line_active = False

def log(tag, msg, level="INFO"):
    global is_progress_line_active
    if is_progress_line_active:
        sys.stdout.write("\n")
        is_progress_line_active = False
    colors = {
        "SYSTEM": "\033[36m",
        "SUCCESS": "\033[32m",
        "INFO": "\033[34m",
        "WARN": "\033[33m",
        "ERROR": "\033[31m",
        "DRIVE": "\033[35m",
        "FETCH": "\033[94m",
        "PROBE": "\033[95m",
        "PROGRESS": "\033[96m",
        "TRANSFER": "\033[33m",
        "EXTRACT": "\033[96m",
        "NOTIFY": "\033[92m"
    }
    reset = "\033[0m"
    dim = "\033[90m"
    ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
    tag_color = colors.get(level, "\033[37m")
    print(f"{dim}{ts}{reset} {tag_color}[{tag}]{reset} {msg}")

def send_telegram(token, chat_id, text):
    if not token.strip() or not chat_id.strip():
        return
    try:
        url = f"https://api.telegram.org/bot{token.strip()}/sendMessage"
        payload = json.dumps({"chat_id": chat_id.strip(), "text": text}).encode('utf-8')
        req = urllib.request.Request(url, data=payload, headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=10) as resp:
            if resp.status == 200:
                log("NOTIFY", "Đã gửi thông báo báo cáo tiến trình qua Telegram.", "NOTIFY")
    except Exception as e:
        log("WARN", f"Không thể gửi thông báo Telegram: {e}", "WARN")

def mask_proxy(p_str):
    return re.sub(r'//.*?:.*?@', '//***:***@', p_str)

def get_unique_filename(dest_dir, filename):
    base, ext = os.path.splitext(filename)
    counter = 1
    new_filename = filename
    while os.path.exists(os.path.join(dest_dir, new_filename)):
        new_filename = f"{base}_{counter}{ext}"
        counter += 1
    return new_filename

# Custom Logger chuẩn hóa thông báo và loại trừ dòng rác
logged_destinations = set()

class YTDLSystemLogger:
    def debug(self, msg):
        global logged_destinations
        if msg.startswith('[download] Destination:'):
            dest_name = os.path.basename(msg.split(': ')[-1])
            # Chỉ in duy nhất 1 lần khi bắt đầu tạo file
            if dest_name not in logged_destinations:
                logged_destinations.add(dest_name)
                log("FETCH", f"Bắt đầu khởi tạo luồng tệp: {dest_name}", "INFO")
        elif 'Merging formats into' in msg:
            merged_name = os.path.basename(msg.split('"')[1]) if '"' in msg else "tệp hoàn chỉnh"
            log("TRANSFER", f"Đang ghép kênh video + âm thanh hoàn chỉnh (FFmpeg): {merged_name}...", "TRANSFER")
            
    def info(self, msg):
        if 'Extracting URL' in msg:
            log("FETCH", "Đang kết nối và phân tích luồng trực tiếp từ máy chủ...", "INFO")
        elif 'Downloading 1 format(s)' in msg:
            fmt = msg.split(': ')[-1] if ': ' in msg else "tối ưu"
            log("FETCH", f"Đã khóa định dạng tải xuống tốt nhất ({fmt}). Bắt đầu truyền tải...", "INFO")
        elif not msg.startswith('[download]') and not msg.startswith('[youtube]'):
            log("INFO", msg, "INFO")
            
    def warning(self, msg):
        # Lọc các cảnh báo nội bộ không ảnh hưởng của yt-dlp để log sạch sẽ
        ignore_keywords = ['No supported JavaScript runtime', 'Remote components', 'n challenge solving', 'PO Token', 'SABR-only', 'URL']
        if any(w in msg for w in ignore_keywords):
            return
        log("WARN", msg, "WARN")
        
    def error(self, msg):
        if 'Got error: 297 bytes read' not in msg:
            log("ERROR", msg, "ERROR")

# Progress Hook cập nhật mượt mà không bị lặp và hiển thị ETA rõ ràng
last_progress_time = 0

def ytdl_progress_hook(d):
    global last_progress_time, is_progress_line_active
    if d['status'] == 'downloading':
        now = time.time()
        if now - last_progress_time >= 1.0:
            last_progress_time = now
            is_progress_line_active = True
            downloaded = d.get('downloaded_bytes', 0)
            total = d.get('total_bytes') or d.get('total_bytes_estimate', 0)
            speed = d.get('speed') or 0
            eta = d.get('eta')
            
            percent_str = f"{(downloaded / total * 100):.1f}%" if total > 0 else "---%"
            speed_str = f"{(speed / (1024**2)):.2f} MB/s" if speed else "--- MB/s"
            
            if eta is not None:
                mins, secs = divmod(int(eta), 60)
                hrs, mins = divmod(mins, 60)
                eta_str = f"{hrs:02d}h:{mins:02d}m:{secs:02d}s" if hrs > 0 else f"{mins:02d}m:{secs:02d}s"
            else:
                eta_str = "Đang tính..."
                
            dl_mb = downloaded / (1024**2)
            tot_mb = total / (1024**2) if total else 0
            size_part = f"{dl_mb:.1f}/{tot_mb:.1f} MB" if tot_mb > 0 else f"{dl_mb:.1f} MB"
            
            ts = datetime.now(VN_TZ).strftime("%H:%M:%S")
            dim = "\033[90m"
            cyan = "\033[96m"
            reset = "\033[0m"
            line = f"{dim}{ts}{reset} {cyan}[PROGRESS]{reset} Tiến độ: {percent_str:<6} ({size_part}) | Tốc độ: {speed_str:<11} | Ước tính còn lại (ETA): {eta_str}"
            sys.stdout.write(f"\r{line}\033[K")
            sys.stdout.flush()
            
    elif d['status'] == 'finished':
        if is_progress_line_active:
            sys.stdout.write("\n")
            is_progress_line_active = False
        fn = os.path.basename(d.get('filename', 'tệp'))
        log("SUCCESS", f"Đã tải xong luồng dữ liệu của tệp: {fn}", "SUCCESS")

temp_workspace = "/content/temp_download"
cache_path = os.path.join(temp_workspace, "session_tasks.json")

if not os.path.exists(cache_path):
    log("ERROR", "Chưa tìm thấy dữ liệu nguồn. Vui lòng chạy ô Bước 3 trước khi thực thi Bước 4.", "ERROR")
    raise FileNotFoundError("Chưa thực thi Bước 3 để nạp liên kết.")

with open(cache_path, "r", encoding="utf-8") as f:
    session_data = json.load(f)

urls = session_data.get("urls", [])
proxies = session_data.get("proxies", [])
referer = session_data.get("referer", "")
cookie_file_path = session_data.get("cookie_file")
tasks_meta = session_data.get("tasks", [])

os.makedirs(THU_MUC_DRIVE, exist_ok=True)

# Kiểm tra bộ đệm máy ảo Colab
_, _, vm_free = shutil.disk_usage('/content')
free_gb = vm_free / (1024**3)
log("SYSTEM", f"Kiểm tra bộ nhớ: Bộ đệm máy ảo Colab khả dụng {free_gb:.2f} GB.", "INFO")
if free_gb < 2.0:
    log("WARN", f"Cảnh báo: Bộ đệm máy ảo Colab còn dưới {free_gb:.2f} GB khả dụng.", "WARN")

log("SYSTEM", f"Bắt đầu thực thi tải xuống cho {len(urls)} tác vụ đa nguồn...", "SYSTEM")

success_count = 0
failed_count = 0
saved_files_report = []

for index, target_url in enumerate(urls, start=1):
    current_meta = tasks_meta[index-1] if index-1 < len(tasks_meta) else {}
    task_type = current_meta.get("type", "standard")
    title_display = current_meta.get("title", target_url)
    res_display = current_meta.get("resolution", "")
    sz_display = current_meta.get("filesize_str", "")
    
    extra_info = f" ({res_display} | {sz_display})" if res_display and res_display != "Không xác định" else ""
    log("FETCH", f"[{index}/{len(urls)}] Đang tải: {title_display}{extra_info}", "FETCH")
    
    current_proxy = proxies[(index - 1) % len(proxies)] if proxies else None
    if current_proxy:
        log("SYSTEM", f"Định tuyến qua proxy ({((index - 1) % len(proxies)) + 1}/{len(proxies)}): {mask_proxy(current_proxy)}", "INFO")
        
    sub_task_dir = os.path.join(temp_workspace, f"download_task_{index:03d}")
    os.makedirs(sub_task_dir, exist_ok=True)
    
    download_ok = False
    
    # 1. NGUỒN GOOGLE DRIVE -> Sử dụng gdown
    if task_type == "gdrive" or 'drive.google.com' in target_url.lower():
        is_folder = '/folders/' in target_url.lower() or current_meta.get("is_folder", False)
        log("FETCH", f"Kích hoạt engine gdown cho Google Drive ({'Thư mục' if is_folder else 'Tệp tin'})...", "FETCH")
        if is_folder:
            cmd = f'gdown --folder "{target_url}" -O "{sub_task_dir}" --remaining-ok'
        else:
            cmd = f'gdown "{target_url}" -O "{sub_task_dir}" --fuzzy'
        if current_proxy:
            cmd = f'export http_proxy="{current_proxy}" && export https_proxy="{current_proxy}" && ' + cmd
        ret = os.system(cmd)
        if ret == 0:
            download_ok = True
        else:
            log("WARN", "Engine gdown báo lỗi. Thử nghiệm tải dự phòng bằng aria2c...", "WARN")
            direct_cmd = f'aria2c -x16 -s16 -k1M --file-allocation=none -d "{sub_task_dir}" "{target_url}"'
            if os.system(direct_cmd) == 0:
                download_ok = True
            else:
                log("ERROR", f"Không thể tải tệp từ Google Drive: {target_url}", "ERROR")

    # 2. NGUỒN MEGA -> Sử dụng megatools
    elif task_type == "mega" or 'mega.nz' in target_url.lower() or 'mega.co.nz' in target_url.lower():
        log("FETCH", "Kích hoạt engine megatools cho nền tảng MEGA...", "FETCH")
        cmd = f'megatools dl "{target_url}" --path "{sub_task_dir}"'
        if os.system(cmd) == 0:
            download_ok = True
        else:
            log("ERROR", f"Không thể tải tệp từ MEGA: {target_url}", "ERROR")

    # 3. NGUỒN BITTORRENT / MAGNET -> Sử dụng aria2c P2P
    elif task_type == "torrent" or target_url.startswith('magnet:?') or target_url.lower().endswith('.torrent'):
        log("FETCH", "Kích hoạt engine aria2c P2P cho BitTorrent/Magnet...", "FETCH")
        trackers = [
            "udp://tracker.opentrackr.org:1337/announce",
            "udp://open.tracker.cl:1337/announce",
            "udp://9.rarbg.com:2810/announce",
            "udp://tracker.openbittorrent.com:6969/announce",
            "http://tracker.openbittorrent.com:80/announce"
        ]
        tracker_arg = f"--bt-tracker={','.join(trackers)}"
        aria2_cmd = (
            f'aria2c --seed-time=0 --summary-interval=3 --file-allocation=none '
            f'--enable-dht=true --dht-listen-port=6881 --bt-enable-lpd=true '
            f'{tracker_arg} --dir="{sub_task_dir}" "{target_url}"'
        )
        if current_proxy:
            aria2_cmd += f' --all-proxy="{current_proxy}"'
        ret = os.system(aria2_cmd)
        if ret == 0:
            download_ok = True
        else:
            log("ERROR", f"Tiến trình aria2c báo lỗi với mã thoát: {ret}", "ERROR")

    # 4. NGUỒN DIRECT LINK FILE -> Sử dụng aria2c đa luồng
    elif task_type == "direct_file":
        log("FETCH", "Kích hoạt engine aria2c tăng tốc tải đa luồng (-x16 -s16)...", "FETCH")
        direct_cmd = f'aria2c -x16 -s16 -k1M --summary-interval=3 --file-allocation=none -d "{sub_task_dir}" "{target_url}"'
        if current_proxy:
            direct_cmd += f' --all-proxy="{current_proxy}"'
        ret = os.system(direct_cmd)
        if ret == 0:
            download_ok = True
        else:
            log("WARN", "aria2c gặp lỗi. Chuyển sang cURL dự phòng...", "WARN")
            out_f = os.path.join(sub_task_dir, os.path.basename(urllib.parse.urlsplit(target_url).path) or "file.bin")
            ret2 = os.system(f'curl -L "{target_url}" -o "{out_f}"')
            if ret2 == 0:
                download_ok = True
            else:
                log("ERROR", f"Không thể tải tệp trực tiếp: {target_url}", "ERROR")

    # 5. NGUỒN VIDEO STREAM / SOCIAL PLATFORMS -> Sử dụng yt-dlp native kèm Live ETA & Deno JS Runtime
    else:
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
            'Accept': '*/*',
            'Accept-Language': 'en-US,en;q=0.9,vi;q=0.8'
        }
        if referer:
            headers['Referer'] = referer
            headers['Origin'] = urllib.parse.urlsplit(referer).scheme + "://" + urllib.parse.urlsplit(referer).netloc
        elif 'seenow' in target_url.lower() or 'vtv' in target_url.lower():
            headers['Referer'] = 'https://vtvprime.vn/'
            headers['Origin'] = 'https://vtvprime.vn'

        if TEN_FILE_TU_DAT.strip():
            custom_raw = TEN_FILE_TU_DAT.strip()
            base_name, specified_ext = os.path.splitext(custom_raw)
            num_suffix = f"_{index}" if len(urls) > 1 else ""
            if specified_ext:
                out_template = os.path.join(sub_task_dir, f"{base_name}{num_suffix}{specified_ext}")
            else:
                out_template = os.path.join(sub_task_dir, f"{base_name}{num_suffix}.%(ext)s")
        else:
            out_template = os.path.join(sub_task_dir, '%(title)s.%(ext)s')

        ydl_opts = {
            'outtmpl': out_template,
            'format': 'bestvideo*+bestaudio/best',
            'merge_output_format': 'mp4',
            'quiet': False,
            'no_warnings': True,
            'retries': 10,
            'fragment_retries': 10,
            'concurrent_fragment_downloads': 4,
            'http_headers': headers,
            'remote_components': ['ejs:github'],
            'logger': YTDLSystemLogger(),
            'progress_hooks': [ytdl_progress_hook],
            'extractor_args': {
                'youtube': {
                    'player_client': ['web', 'mweb', 'android']
                }
            }
        }
        if cookie_file_path:
            ydl_opts['cookiefile'] = cookie_file_path
        if current_proxy:
            ydl_opts['proxy'] = current_proxy

        log("FETCH", "Khởi chạy tải luồng ổn định tốc độ cao...", "INFO")
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([target_url])
            download_ok = True
        except Exception as err:
            log("WARN", f"Định dạng ghép kênh gặp lỗi ({err}). Tự động thử định dạng tương thích...", "WARN")
            ydl_opts['extractor_args']['youtube']['player_client'] = ['android']
            ydl_opts['format'] = 'bestvideo+bestaudio/best'
            try:
                with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                    ydl.download([target_url])
                download_ok = True
            except Exception as err2:
                log("WARN", f"Chuyển sang định dạng tương thích đơn luồng 'best'...", "WARN")
                ydl_opts['format'] = 'best'
                try:
                    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                        ydl.download([target_url])
                    download_ok = True
                except Exception as err3:
                    log("ERROR", f"Không thể tải xuống tệp từ nguồn: {target_url} ({err3})", "ERROR")

    # Giải nén tệp tự động nếu được bật
    archive_extensions = ('.zip', '.rar', '.7z', '.tar', '.gz', '.tar.gz', '.tgz', '.bz2')
    if download_ok and TU_DONG_GIAI_NEN:
        for root, dirs, files in os.walk(sub_task_dir):
            for f in files:
                f_path = os.path.join(root, f)
                lower_f = f.lower()
                if lower_f.endswith(archive_extensions):
                    folder_name = os.path.splitext(f)[0]
                    if folder_name.lower().endswith('.tar'):
                        folder_name = os.path.splitext(folder_name)[0]
                    extract_dest = os.path.join(sub_task_dir, f"extracted_{folder_name}")
                    os.makedirs(extract_dest, exist_ok=True)
                    log("EXTRACT", f"Đang tiến hành giải nén tệp lưu trữ: {f}...", "EXTRACT")
                    cmd_extract = f'7z x -y "{f_path}" -o"{extract_dest}" > /dev/null 2>&1'
                    status_ext = os.system(cmd_extract)
                    if status_ext == 0:
                        log("SUCCESS", f"Giải nén thành công tệp: {f}", "SUCCESS")
                        try:
                            os.remove(f_path)
                        except:
                            pass
                    else:
                        log("WARN", f"Không thể giải nén tự động tệp: {f}", "WARN")

    # Quét tệp tin đầu ra và chuyển giao lên Google Drive (Hỗ trợ cả file và folder)
    output_items = []
    for entry in os.listdir(sub_task_dir):
        full_entry = os.path.join(sub_task_dir, entry)
        if not entry.endswith(('.part', '.ytdl', '.aria2')):
            output_items.append(full_entry)

    if download_ok and output_items:
        log("TRANSFER", f"Chuyển giao {len(output_items)} mục dữ liệu vào Google Drive: {THU_MUC_DRIVE}", "TRANSFER")
        
        custom_raw = TEN_FILE_TU_DAT.strip()
        custom_base, specified_ext = os.path.splitext(custom_raw) if custom_raw else ("", "")
        num_suffix = f"_{index}" if len(urls) > 1 else ""

        for item_idx, item_path in enumerate(output_items):
            orig_name = os.path.basename(item_path)
            orig_base, orig_ext = os.path.splitext(orig_name)

            if os.path.isdir(item_path):
                dest_name = f"{custom_base}{num_suffix}" if custom_raw else orig_name
                target_dest_path = os.path.join(THU_MUC_DRIVE, dest_name)
                if os.path.exists(target_dest_path):
                    target_dest_path = os.path.join(THU_MUC_DRIVE, f"{dest_name}_{item_idx+1}")
                shutil.move(item_path, target_dest_path)
                log("SUCCESS", f"Lưu trữ thành công thư mục: {target_dest_path}", "SUCCESS")
                saved_files_report.append(os.path.basename(target_dest_path) + " (Thư mục)")
            else:
                if custom_raw:
                    final_ext = specified_ext if specified_ext else orig_ext
                    part_suffix = f"_{item_idx+1}" if len(output_items) > 1 else ""
                    dest_filename = f"{custom_base}{num_suffix}{part_suffix}{final_ext}"
                else:
                    dest_filename = orig_name

                target_fname = get_unique_filename(THU_MUC_DRIVE, dest_filename)
                dest_path = os.path.join(THU_MUC_DRIVE, target_fname)
                shutil.move(item_path, dest_path)
                log("SUCCESS", f"Lưu trữ thành công: {dest_path}", "SUCCESS")
                saved_files_report.append(target_fname)
                
        success_count += 1
    else:
        log("ERROR", f"Thất bại khi xử lý tiến trình [{index}/{len(urls)}]", "ERROR")
        failed_count += 1

try:
    shutil.rmtree(temp_workspace, ignore_errors=True)
except:
    pass

print("\n" + "="*60)
log("SYSTEM", f"Tổng kết: {success_count} thành công, {failed_count} thất bại.", "SYSTEM")
log("DRIVE", f"Thư mục lưu trữ trên Google Drive: {THU_MUC_DRIVE}", "DRIVE")
print("="*60)

# Gửi thông báo hoàn tất qua Telegram Bot nếu có cấu hình
if TELEGRAM_BOT_TOKEN.strip() and TELEGRAM_CHAT_ID.strip():
    ts_now = datetime.now(VN_TZ).strftime("%H:%M:%S - %d/%m/%Y")
    report_lines = [
        "[Colab Drive] Bao cao ket qua tien trinh:",
        f"- Thoi gian hoan tat: {ts_now}",
        f"- Tong so URL: {len(urls)}",
        f"- Thanh cong: {success_count}",
        f"- That bai: {failed_count}",
        f"- Thu muc luu tru: {THU_MUC_DRIVE}"
    ]
    if saved_files_report:
        report_lines.append(f"- Tep/Thu muc da luu ({len(saved_files_report)}):")
        for fn in saved_files_report[:10]:
            report_lines.append(f"  + {fn}")
        if len(saved_files_report) > 10:
            report_lines.append(f"  ... va {len(saved_files_report) - 10} muc khac")
    
    send_telegram(TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID, "\n".join(report_lines))

